### MCP servers are useful for exposing tools to an LLM 
i.e for discovery and execution of said tools on the MCP serverside
to be connected to agents 
wihtout needeing to re implement for different agents etc


In this project we will implement two seperate MCP servers 

One for out items metadata tool

One for our reviews tool 

We deploy these sevrers as docker apps in the docker compose file

```
cd apps
uv init --package items_mcp_server
uv add --package items_mcp_server fastmcp openai pydantic pydantic-settings qdrant-client cohere 
```


we will use an application called fast mcp to make these mcp servers

please see 
./apps/items_mcp_server 
./apps/reviews_mcp_server 

Now we will learn how to connect the mcp servers to our agentic workflow -> agenet -> langgrpah graph

we will use langchians helpers for this

mainly we will add the funcitonality on top of the graph 

### Import deps

In [1]:
from fastmcp import Client

from pydantic import BaseModel

import cohere

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langsmith import traceable, get_current_run_tree

import instructor

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Send

from langchain_core.messages import SystemMessage, convert_to_openai_messages, HumanMessage, AIMessage
from IPython.display import Image, display

from typing import Literal, Dict, Any, Annotated, List
from pydantic import Field
from operator import add

import random
import openai
import pandas as pd

from jinja2 import Template

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

the same lib thats been used to make the mcp servers can be used as the mcp client as well

uv add --dev fastmcp

### List available tools in MCP servers on http://localhost:8001/mcp and http://localhost:8002/mcp

note this is an async client

In [2]:
client = Client("http://localhost:8001/mcp")

In [3]:
async with client:

    tools = await client.list_tools()

In [4]:
tools

[Tool(name='get_formatted_item_context', title=None, description='Search available products and return the top k matching inventory items.\n\nExpand the customer\'s question into 1-5 concise search statements and issue them\nin parallel in a single turn. Each statement covers one distinct product or\nattribute; no two may express the same intent. Use natural product-description\nlanguage. If no brand or model is specified, search broadly rather than refusing.\n\n    "Earphones for me and a waterproof speaker"\n        -> "Personal earphones" | "Waterproof speaker"\n    "A warm winter jacket for hiking"\n        -> "Insulated winter jacket" | "Hiking outerwear for cold weather"\n\nBefore calling, check what earlier calls in this conversation already returned.\nSearch only for what is missing; results already retrieved remain valid and must\nnot be fetched again.', inputSchema={'additionalProperties': False, 'properties': {'query': {'type': 'string', 'description': 'A single search state

In [5]:
print("======NAME=======")
print(tools[0].name)
print("======DESCRIPTION=======")
print(tools[0].description)
print("======INPUT SCHEMA=======")
print(tools[0].inputSchema)

======NAME=======
get_formatted_item_context
======DESCRIPTION=======
Search available products and return the top k matching inventory items.

Expand the customer's question into 1-5 concise search statements and issue them
in parallel in a single turn. Each statement covers one distinct product or
attribute; no two may express the same intent. Use natural product-description
language. If no brand or model is specified, search broadly rather than refusing.

    "Earphones for me and a waterproof speaker"
        -> "Personal earphones" | "Waterproof speaker"
    "A warm winter jacket for hiking"
        -> "Insulated winter jacket" | "Hiking outerwear for cold weather"

Before calling, check what earlier calls in this conversation already returned.
Search only for what is missing; results already retrieved remain valid and must
not be fetched again.
======INPUT SCHEMA=======
{'additionalProperties': False, 'properties': {'query': {'type': 'string', 'description': 'A single search stat

Note its one client per server

In [6]:
client = Client("http://localhost:8002/mcp")

In [7]:
tools

[Tool(name='get_formatted_item_context', title=None, description='Search available products and return the top k matching inventory items.\n\nExpand the customer\'s question into 1-5 concise search statements and issue them\nin parallel in a single turn. Each statement covers one distinct product or\nattribute; no two may express the same intent. Use natural product-description\nlanguage. If no brand or model is specified, search broadly rather than refusing.\n\n    "Earphones for me and a waterproof speaker"\n        -> "Personal earphones" | "Waterproof speaker"\n    "A warm winter jacket for hiking"\n        -> "Insulated winter jacket" | "Hiking outerwear for cold weather"\n\nBefore calling, check what earlier calls in this conversation already returned.\nSearch only for what is missing; results already retrieved remain valid and must\nnot be fetched again.', inputSchema={'additionalProperties': False, 'properties': {'query': {'type': 'string', 'description': 'A single search state

In [8]:
print("======NAME=======")
print(tools[0].name)
print("======DESCRIPTION=======")
print(tools[0].description)
print("======INPUT SCHEMA=======")
print(tools[0].inputSchema)

======NAME=======
get_formatted_item_context
======DESCRIPTION=======
Search available products and return the top k matching inventory items.

Expand the customer's question into 1-5 concise search statements and issue them
in parallel in a single turn. Each statement covers one distinct product or
attribute; no two may express the same intent. Use natural product-description
language. If no brand or model is specified, search broadly rather than refusing.

    "Earphones for me and a waterproof speaker"
        -> "Personal earphones" | "Waterproof speaker"
    "A warm winter jacket for hiking"
        -> "Insulated winter jacket" | "Hiking outerwear for cold weather"

Before calling, check what earlier calls in this conversation already returned.
Search only for what is missing; results already retrieved remain valid and must
not be fetched again.
======INPUT SCHEMA=======
{'additionalProperties': False, 'properties': {'query': {'type': 'string', 'description': 'A single search stat

### Executing one of the tools

In [10]:
client = Client("http://localhost:8001/mcp")

In [11]:
async with client:

    result = await client.call_tool("get_formatted_item_context", {"query": "What kind of earphones can I get?", "top_k": 10})

ToolError: Error calling tool 'get_formatted_item_context': Unexpected Response: 404 (Not Found)
Raw response content:
b'{"status":{"error":"Not found: Collection `Amazon-items-collection-01-hybrid-search` doesn\'t exist!"},"time":0.007400208}'

In [12]:
result

NameError: name 'result' is not defined

In [ ]:
print(result.content[0].text)

### MCP Tool Calling via LangChain

uv add --dev langchain-mcp-adapters

In [14]:
from langchain_mcp_adapters.client import MultiServerMCPClient

async client connecting to multiple servers

In [27]:
client = MultiServerMCPClient(
    {
        "items_mcp_server": {
            "url": "http://localhost:8001/mcp",
            "transport": "http"
        },
        "reviews_mcp_server": {
            "url": "http://localhost:8002/mcp",
            "transport": "http"
        },
    }
)

list all tools in all servers

In [28]:
mcp_tools = await client.get_tools()

In [29]:
mcp_tools

[StructuredTool(name='get_formatted_item_context', description='Search available products and return the top k matching inventory items.\n\nExpand the customer\'s question into 1-5 concise search statements and issue them\nin parallel in a single turn. Each statement covers one distinct product or\nattribute; no two may express the same intent. Use natural product-description\nlanguage. If no brand or model is specified, search broadly rather than refusing.\n\n    "Earphones for me and a waterproof speaker"\n        -> "Personal earphones" | "Waterproof speaker"\n    "A warm winter jacket for hiking"\n        -> "Insulated winter jacket" | "Hiking outerwear for cold weather"\n\nBefore calling, check what earlier calls in this conversation already returned.\nSearch only for what is missing; results already retrieved remain valid and must\nnot be fetched again.', args_schema={'additionalProperties': False, 'properties': {'query': {'type': 'string', 'description': 'A single search stateme

each of these tools is a structured tool -> so is langchain compatible

this applys to all tools with @tool decorator

In [30]:
result = await mcp_tools[0].ainvoke({"query": "What kind of earphones can I get?", "top_k": 10})

In [31]:
result

[{'type': 'text',
  'text': 'Error calling tool \'get_formatted_item_context\': Unexpected Response: 404 (Not Found)\nRaw response content:\nb\'{"status":{"error":"Not found: Collection `Amazon-items-collection-01-hybrid-search` doesn\\\'t exist!"},"time":0.000060959}\'',
  'id': 'lc_7d52375f-4328-4407-9325-3fccc5dbd12d'}]

In [ ]:
print(result[0]["text"])

Error calling tool 'get_formatted_item_context': Unexpected Response: 404 (Not Found)
Raw response content:
b'{"status":{"error":"Not found: Collection `Amazon-items-collection-01-hybrid-search` doesn\'t exist!"},"time":0.000060959}'


In [34]:
print(result[0]["text"])

Error calling tool 'get_formatted_item_context': Unexpected Response: 404 (Not Found)
Raw response content:
b'{"status":{"error":"Not found: Collection `Amazon-items-collection-01-hybrid-search` doesn\'t exist!"},"time":0.000060959}'
